# StyleTTS2 Valenciano - Test de Reconstrucción Stage 1

Stage 1 entrena: `text_encoder`, `style_encoder`, `decoder`, `text_aligner`, `pitch_extractor`

Stage 1 **NO** entrena: `predictor` (duración/F0), `bert`, `diffusion`

Por lo tanto, no se puede hacer TTS desde texto directamente. Lo que sí se puede hacer es un **test de reconstrucción**:
dado un audio original, extraer su F0, alignment y estilo, y reconstruirlo a través del decoder.

Si la reconstrucción suena bien, Stage 1 funciona correctamente y el modelo está listo para Stage 2.

In [1]:
import torch
torch.manual_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

import random
random.seed(0)

import numpy as np
np.random.seed(0)

In [2]:
%cd ..

/home/calia/styletts2-valenciano/StyleTTS2


In [3]:
import time
import yaml
from munch import Munch
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import torchaudio
import librosa
from nltk.tokenize import word_tokenize
import IPython.display as ipd

from models import *
from utils import *
from text_utils import TextCleaner
textclenaer = TextCleaner()

%matplotlib inline

177


In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cuda


In [5]:
to_mel = torchaudio.transforms.MelSpectrogram(
    n_mels=80, n_fft=2048, win_length=1200, hop_length=300)
mean, std = -4, 4

def preprocess(wave):
    wave_tensor = torch.from_numpy(wave).float()
    mel_tensor = to_mel(wave_tensor)
    mel_tensor = (torch.log(1e-5 + mel_tensor.unsqueeze(0)) - mean) / std
    return mel_tensor

def length_to_mask(lengths):
    mask = torch.arange(lengths.max()).unsqueeze(0).expand(lengths.shape[0], -1).type_as(lengths)
    mask = torch.gt(mask+1, lengths.unsqueeze(1))
    return mask

### Cargar modelo

In [6]:
import phonemizer
global_phonemizer = phonemizer.backend.EspeakBackend(
    language='ca-va',
    preserve_punctuation=True,
    with_stress=True,
    language_switch='remove-flags',
)

In [7]:
config = yaml.safe_load(open('Models/LJSpeech/config.yml'))

ASR_config = config.get('ASR_config', False)
ASR_path = config.get('ASR_path', False)
text_aligner = load_ASR_models(ASR_path, ASR_config)

F0_path = config.get('F0_path', False)
pitch_extractor = load_F0_models(F0_path)

from Utils.PLBERT.util import load_plbert
BERT_path = config.get('PLBERT_dir', False)
plbert = load_plbert(BERT_path)

/home/calia/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando PL-BERT desde: Utils/PLBERT/


In [8]:
model = build_model(recursive_munch(config['model_params']), text_aligner, pitch_extractor, plbert)
_ = [model[key].eval() for key in model]
_ = [model[key].to(device) for key in model]

/home/calia/.venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/calia/.venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


In [9]:
checkpoint_path = 'Models/LJSpeech/first_stage.pth'  # <-- Tu checkpoint de Stage 1

params_whole = torch.load(checkpoint_path, map_location='cpu')
params = params_whole['net']

print(f'Checkpoint: {checkpoint_path}')
print(f'Epoch: {params_whole.get("epoch", "?")}')
print(f'Val loss: {params_whole.get("val_loss", "?")}')
print()

for key in model:
    if key in params:
        print(f'{key} loaded')
        try:
            model[key].load_state_dict(params[key])
        except:
            from collections import OrderedDict
            state_dict = params[key]
            new_state_dict = OrderedDict()
            for k, v in state_dict.items():
                name = k[7:]  # remove `module.`
                new_state_dict[name] = v
            model[key].load_state_dict(new_state_dict, strict=False)
    else:
        print(f'{key} NOT in checkpoint')

_ = [model[key].eval() for key in model]

Checkpoint: Models/LJSpeech/first_stage.pth
Epoch: 99
Val loss: 0.2665151327171109

bert loaded
bert_encoder loaded
predictor loaded
decoder loaded
text_encoder loaded
predictor_encoder loaded
style_encoder loaded
diffusion loaded
text_aligner loaded
pitch_extractor loaded
mpd loaded
msd loaded
wd loaded


### Test de Reconstrucción

Replica lo que hace `train_first.py` durante el entrenamiento:
1. Carga audio y texto
2. Extrae mel spectrogram
3. Text aligner computa alignment texto-audio
4. Pitch extractor extrae F0 real
5. Style encoder extrae estilo
6. Decoder reconstruye el audio

Si el audio reconstruido suena similar al original, Stage 1 funciona.

In [ ]:
n_down = model.text_aligner.n_down

def reconstruction_test(audio_path, text):
    """
    Test de reconstrucción para Stage 1.
    Usa F0, alignment y estilo del audio original para reconstruir.
    """
    # 1. Cargar y preprocesar audio
    wave, sr = librosa.load(audio_path, sr=24000)
    audio, _ = librosa.effects.trim(wave, top_db=30)
    
    # Padding como hace el dataloader
    audio = np.concatenate([np.zeros([5000]), audio, np.zeros([5000])], axis=0)
    
    mel = preprocess(audio).to(device)  # [1, 80, T]
    
    # 2. Fonemizar texto
    ps = global_phonemizer.phonemize([text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)
    tokens = textclenaer(ps)
    tokens.insert(0, 0)
    tokens.append(0)
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
    
    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)
        
        mel_input_length = torch.LongTensor([mel.shape[-1]]).to(device)
        
        # 3. Text aligner: firma = forward(x, src_key_padding_mask, text_input)
        mask = length_to_mask(mel_input_length // (2 ** n_down)).to(device)
        ppgs, s2s_pred, s2s_attn = model.text_aligner(mel, mask, tokens)
        
        # Procesar attention (igual que en train_first.py)
        s2s_attn = s2s_attn.transpose(-1, -2)
        s2s_attn = s2s_attn[..., 1:]
        s2s_attn = s2s_attn.transpose(-1, -2)
        
        # Attention mask
        attn_mask = (~mask).unsqueeze(-1).expand(mask.shape[0], mask.shape[1], text_mask.shape[-1]).float().transpose(-1, -2)
        attn_mask = attn_mask.float() * (~text_mask).unsqueeze(-1).expand(text_mask.shape[0], text_mask.shape[1], mask.shape[-1]).float()
        attn_mask = (attn_mask < 1)
        s2s_attn.masked_fill_(attn_mask, 0.0)
        
        # Monotonic alignment
        mask_ST = mask_from_lens(s2s_attn, input_lengths, mel_input_length // (2 ** n_down))
        s2s_attn_mono = maximum_path(s2s_attn, mask_ST)
        
        # 4. Text encoder + alignment
        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        asr = (t_en @ s2s_attn_mono)
        
        # 5. Extraer F0 real y norma del mel (log_norm viene de utils.py)
        real_norm = log_norm(mel.unsqueeze(1)).squeeze(1)
        F0_real, _, _ = model.pitch_extractor(mel.unsqueeze(1))
        
        # 6. Extraer estilo
        s = model.style_encoder(mel.unsqueeze(1))
        
        # 7. Decoder: reconstruir audio
        y_rec = model.decoder(asr, F0_real, real_norm, s)
        
    return y_rec.squeeze().cpu().numpy(), audio

### Probar reconstrucción

Cambia `audio_path` a un .wav del dataset y `text` a su transcripción.

In [13]:
# Ejemplo: pon la ruta a un audio del dataset y su texto
audio_path = '/media/datos1/tts-valenciano/styletts2_corts_dataset/wavs/1f046b110282aa31e9bb2f14a6aec1cc.wav'  # <-- CAMBIAR
text = 'transcripción del audio aquí'  # <-- CAMBIAR al texto correspondiente

reconstructed, original = reconstruction_test(audio_path, text)

print(f'Duración original: {len(original)/24000:.2f}s')
print(f'Duración reconstruida: {len(reconstructed)/24000:.2f}s')
print()
print('Original:')
display(ipd.Audio(original, rate=24000))
print('Reconstruido:')
display(ipd.Audio(reconstructed, rate=24000))

ImportError: cannot import name 'log_norm' from 'losses' (/home/calia/styletts2-valenciano/StyleTTS2/losses.py)

### Test con líneas del dataset

Lee directamente del train_list.txt para tener audio + texto automáticamente.

In [ ]:
# Cargar algunos ejemplos del dataset
root_path = '/media/datos1/tts-valenciano/styletts2_corts_dataset/'  # <-- CAMBIAR si es necesario

import os
with open(os.path.join(root_path, 'train_list.txt'), 'r') as f:
    lines = f.readlines()

# Tomar 5 muestras aleatorias
import random
random.seed(42)
samples = random.sample(lines, min(5, len(lines)))

for line in samples:
    parts = line.strip().split('|')
    wav_path = os.path.join(root_path, parts[0])
    phonemes = parts[1]
    
    print(f'Archivo: {parts[0]}')
    print(f'Fonemas: {phonemes[:60]}...')
    
    # Para reconstrucción usamos los fonemas directamente (ya fonemizado)
    # Necesitamos el texto original, pero podemos usar los fonemas
    # En este caso, cargamos el audio y usamos el text aligner
    try:
        wave, sr = librosa.load(wav_path, sr=24000)
        audio, _ = librosa.effects.trim(wave, top_db=30)
        audio = np.concatenate([np.zeros([5000]), audio, np.zeros([5000])], axis=0)
        mel = preprocess(audio).to(device)
        
        # Usar fonemas directamente
        tokens = textclenaer(phonemes)
        tokens.insert(0, 0)
        tokens.append(0)
        tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)
        
        with torch.no_grad():
            input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
            text_mask = length_to_mask(input_lengths).to(device)
            mel_input_length = torch.LongTensor([mel.shape[-1]]).to(device)
            
            _, _, s2s_attn = model.text_aligner(mel, mask=None, input_lengths=mel_input_length)
            s2s_attn = s2s_attn.transpose(-1, -2)[..., 1:].transpose(-1, -2)
            
            mask = length_to_mask(mel_input_length // 4)
            attn_mask = (~mask).unsqueeze(-1).expand(mask.shape[0], mask.shape[1], text_mask.shape[-1]).float().transpose(-1, -2)
            attn_mask = attn_mask.float() * (~text_mask).unsqueeze(-1).expand(text_mask.shape[0], text_mask.shape[1], mask.shape[-1]).float()
            attn_mask = (attn_mask < 1)
            s2s_attn.masked_fill_(attn_mask, 0.0)
            
            mask_ST = mask_from_lens(s2s_attn, input_lengths, mel_input_length // 4)
            s2s_attn_mono = maximum_path(s2s_attn, mask_ST)
            
            t_en = model.text_encoder(tokens, input_lengths, text_mask)
            asr = (t_en @ s2s_attn_mono)
            
            from losses import log_norm
            real_norm = log_norm(mel.unsqueeze(1)).squeeze(1)
            F0_real, _, _ = model.pitch_extractor(mel.unsqueeze(1))
            s = model.style_encoder(mel.unsqueeze(1))
            
            y_rec = model.decoder(asr, F0_real, real_norm, s)
        
        rec = y_rec.squeeze().cpu().numpy()
        print(f'  Original: {len(audio)/24000:.2f}s | Reconstruido: {len(rec)/24000:.2f}s')
        print('  Original:')
        display(ipd.Audio(audio, rate=24000))
        print('  Reconstruido:')
        display(ipd.Audio(rec, rate=24000))
    except Exception as e:
        print(f'  Error: {e}')
    print()